In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
# !hf auth login

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

* Figure out which layers to replace
* write function like LoRA linear for attenton and mha modules
* write function to replce this model layers to above written lora modules while keeping original weights frozen
* make this all into a moddule for the math

### LoRA from Scratch

In [ ]:
import math
import torch
from torch import nn

In [ ]:
layer = nn.Linear(in_features=3, out_features=2, bias=True)
input_tensor = torch.tensor([1., 2., 3.])
input_tensor

tensor([1., 2., 3.])

In [ ]:
with torch.no_grad():
  layer.weight = nn.Parameter(torch.tensor([[0.1, 0.2, 0.3],
                                            [0.4, 0.5, 0.6]]))
  layer.bias = nn.Parameter(torch.tensor([0.7, 0.8]))

In [ ]:
class LoRALinear(nn.Module):
  def __init__(self, base:nn.Linear, r:int, alpha:float = 16.0):
    super().__init__()
    self.r = r
    self.alpha = alpha
    self.base = base
    self.scaling = alpha / r

    # original W (frozen)
    self.base.requires_grad_(False)
    if self.base.bias is not None:
      self.base.bias.requires_grad_(False)

    self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
    self.lora_B = nn.Parameter(torch.empty(base.out_features, r))

    nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    nn.init.zeros_(self.lora_B)


  def foward(self, x):
    base_output = self.base(x)    # Wx
    lora_update = F.linear(F.linear(x, self.lora_A), self.lora_B)

    return base_output + (lora_update * self.scaling)

### DoRA from scrath - same as LoRA but weight decomposition into magnitude and direction components

Dora wight scaling -> ``` W_dora = (m / ||W + ΔW||) * (W + ΔW) ```

In [ ]:
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float = 16.0):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.base = base
        self.scaling = alpha / r

        # Freeze original weights and bias
        self.base.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)

        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.empty(base.out_features, r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        W = base.weight.detach()
        self.magnitude = nn.Parameter(W.norm(p=2, dim=1, keepdim=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Adapted weight = W + LoRA update
        W = self.base.weight
        lora_update = (self.lora_B @ self.lora_A) * self.scaling

        W_adapted = W + lora_update
        W_norm = W_adapted.norm(p=2, dim=1, keepdim=True)
        W_dora = (self.magnitude / W_norm) * W_adapted

        return F.linear(x, W_dora, self.base.bias)

#### Making the class to use DoRA instead of LLama GQA

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from transformers.models.llama.modeling_llama import LlamaAttention, apply_rotary_pos_emb
from transformers import LlamaForCausalLM

In [ ]:
from transformers.models.llama.modeling_llama import LlamaAttention
from typing import Optional

class DoRALlamaMHA(nn.Module):

    def __init__(self, original_attn, rotary_emb, r: int, alpha: float = 16.0):
        super().__init__()

        cfg = original_attn.config
        self.hidden_size   = cfg.hidden_size
        self.num_heads     = cfg.num_attention_heads
        self.num_kv_heads  = cfg.num_key_value_heads
        self.head_dim      = cfg.hidden_size // cfg.num_attention_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads

        self.q_proj = DoRALinear(original_attn.q_proj, r, alpha)
        self.k_proj = DoRALinear(original_attn.k_proj, r, alpha)
        self.v_proj = DoRALinear(original_attn.v_proj, r, alpha)
        self.o_proj = DoRALinear(original_attn.o_proj, r, alpha)

        self.rotary_emb = rotary_emb

    @staticmethod
    def _repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
        if n_rep == 1:
            return x
        B, num_kv_heads, S, head_dim = x.shape
        return (
            x[:, :, None, :, :]
            .expand(B, num_kv_heads, n_rep, S, head_dim)
            .reshape(B, num_kv_heads * n_rep, S, head_dim)
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value=None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ):
        B, S, _ = hidden_states.shape

        Q = self.q_proj(hidden_states)
        K = self.k_proj(hidden_states)
        V = self.v_proj(hidden_states)

        Q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rotary_emb(V, position_ids)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            K, V = past_key_value.update(K, V, self.layer_idx, cache_kwargs)

        K = self._repeat_kv(K, self.num_kv_groups)
        V = self._repeat_kv(V, self.num_kv_groups)

        attn_output = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=(attention_mask is None),
        )

        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output, None, past_key_value

#### Replace llama gqa with dora dqa and see changes

In [ ]:
def apply_dora_to_llama(model, r: int = 16, alpha: float = 32.0):
    for layer_idx, layer in enumerate(model.model.layers):
        rotary_emb = getattr(layer.self_attn, "rotary_emb", None) or getattr(layer, "rotary_emb", None)
        dora_attn = DoRALlamaMHA(layer.self_attn, rotary_emb, r=r, alpha=alpha)
        dora_attn.layer_idx = layer_idx
        layer.self_attn = dora_attn
    return model

model = LlamaForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B",
    torch_dtype=torch.float16,
    device_map="auto",
)
model = apply_dora_to_llama(model, r=16, alpha=32.0)
model

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): DoRALlamaMHA(
          (q_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (k_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=1024, bias=False)
          )
          (v_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=1024, bias=False)
          )
          (o_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=4096, bias=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_l

### Modularizing for convinence

In [44]:
%%writefile /content/drive/MyDrive/DoRA/src/dora.py
import math
from typing import Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaForCausalLM
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb


class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float = 16.0):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.base = base
        self.scaling = alpha / r

        self.base.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)

        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.empty(base.out_features, r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        W = base.weight.detach()
        self.magnitude = nn.Parameter(W.norm(p=2, dim=1, keepdim=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        W = self.base.weight
        lora_update = (self.lora_B @ self.lora_A) * self.scaling
        W_adapted = W + lora_update

        W_norm = W_adapted.norm(p=2, dim=1, keepdim=True)
        W_dora = (self.magnitude / W_norm) * W_adapted

        return F.linear(x, W_dora, self.base.bias)


class DoRALlamaMHA(nn.Module):
    def __init__(self, original_attn, rotary_emb, r: int, alpha: float = 16.0):
        super().__init__()

        cfg = original_attn.config
        self.hidden_size = cfg.hidden_size
        self.num_heads = cfg.num_attention_heads
        self.num_kv_heads = cfg.num_key_value_heads
        self.head_dim = cfg.hidden_size // cfg.num_attention_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads

        self.q_proj = DoRALinear(original_attn.q_proj, r, alpha)
        self.k_proj = DoRALinear(original_attn.k_proj, r, alpha)
        self.v_proj = DoRALinear(original_attn.v_proj, r, alpha)
        self.o_proj = DoRALinear(original_attn.o_proj, r, alpha)

        self.rotary_emb = rotary_emb
        self.layer_idx: Optional[int] = getattr(original_attn, "layer_idx", None)

    @staticmethod
    def _repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
        if n_rep == 1:
            return x
        B, num_kv_heads, S, head_dim = x.shape
        return (
            x[:, :, None, :, :]
            .expand(B, num_kv_heads, n_rep, S, head_dim)
            .reshape(B, num_kv_heads * n_rep, S, head_dim)
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value=None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ):
        B, S, _ = hidden_states.shape

        Q = self.q_proj(hidden_states)
        K = self.k_proj(hidden_states)
        V = self.v_proj(hidden_states)

        Q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rotary_emb(V, position_ids)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            K, V = past_key_value.update(K, V, self.layer_idx, cache_kwargs)

        K = self._repeat_kv(K, self.num_kv_groups)
        V = self._repeat_kv(V, self.num_kv_groups)

        attn_output = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=(attention_mask is None),
        )

        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output, None


def apply_dora_to_llama(model: LlamaForCausalLM, r: int = 16, alpha: float = 32.0):
    rotary_emb = (
        getattr(model.model, "rotary_emb", None) or
        getattr(model.model.layers[0].self_attn, "rotary_emb", None) or
        getattr(model.model.layers[0], "rotary_emb", None)
    )
    for layer_idx, layer in enumerate(model.model.layers):
        dora_attn = DoRALlamaMHA(layer.self_attn, rotary_emb, r=r, alpha=alpha)
        dora_attn.layer_idx = layer_idx
        layer.self_attn = dora_attn
    return model


def dora_trainable_params(model) -> List[nn.Parameter]:
    return [p for n, p in model.named_parameters() if p.requires_grad]


def dora_param_count(model) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total": total,
        "trainable": trainable,
        "frozen": total - trainable,
        "trainable_pct": round(100 * trainable / total, 4),
    }

Writing /content/dora.py


In [45]:
%%writefile /content/drive/MyDrive/DoRA/src/test_dora.py

import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, "/content/drive/MyDrive/DoRA/src")
from dora import DoRALinear, DoRALlamaMHA, apply_dora_to_llama, dora_param_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Running on {DEVICE} ({DTYPE})\n")

PASS = "✅ PASS"
FAIL = "❌ FAIL"


def test_trainable_params():
    base  = nn.Linear(64, 128, bias=True)
    layer = DoRALinear(base, r=4, alpha=8.0)

    frozen    = [n for n, p in layer.named_parameters() if not p.requires_grad]
    trainable = [n for n, p in layer.named_parameters() if p.requires_grad]

    ok = (set(frozen) == {"base.weight", "base.bias"}) and (set(trainable) == {"lora_A", "lora_B", "magnitude"})
    print(f"TEST 1 — Trainable/frozen split       {PASS if ok else FAIL}")
    if not ok:
        print(f"  frozen={frozen}  trainable={trainable}")

test_trainable_params()


def test_identity_at_init():
    torch.manual_seed(0)
    base     = nn.Linear(64, 128, bias=False)
    layer    = DoRALinear(base, r=4, alpha=8.0).to(DTYPE)
    x        = torch.randn(2, 10, 64, dtype=DTYPE)
    out_dora = layer(x)
    out_base = F.linear(x, base.weight.to(DTYPE))
    ok       = torch.allclose(out_dora, out_base, atol=1e-4)
    print(f"TEST 2 — Identity at init             {PASS if ok else FAIL}")
    if not ok:
        print(f"  max_diff={(out_dora - out_base).abs().max().item():.6f}")

test_identity_at_init()


def test_magnitude_init():
    base          = nn.Linear(64, 128, bias=False)
    layer         = DoRALinear(base, r=4, alpha=8.0)
    expected_vals = base.weight.detach().norm(p=2, dim=1, keepdim=True)
    ok = (layer.magnitude.shape == (128, 1)) and torch.allclose(layer.magnitude.data, expected_vals, atol=1e-6)
    print(f"TEST 3 — Magnitude init shape/value   {PASS if ok else FAIL}")

test_magnitude_init()


def test_output_shape():
    B, S, IN, OUT = 2, 7, 64, 128
    base  = nn.Linear(IN, OUT)
    layer = DoRALinear(base, r=8).to(DTYPE)
    x     = torch.randn(B, S, IN, dtype=DTYPE)
    out   = layer(x)
    ok    = (out.shape == (B, S, OUT))
    print(f"TEST 4 — Output shape {str(tuple(out.shape)):<20} {PASS if ok else FAIL}")

test_output_shape()


def test_gradients():
    base  = nn.Linear(32, 64)
    layer = DoRALinear(base, r=4).to(torch.float32)
    x     = torch.randn(2, 5, 32)
    layer(x).sum().backward()
    ok = (layer.lora_A.grad is not None and
          layer.lora_B.grad is not None and
          layer.magnitude.grad is not None and
          base.weight.grad is None)
    print(f"TEST 5 — Gradients (train/frozen)     {PASS if ok else FAIL}")

test_gradients()


def test_unit_direction():
    torch.manual_seed(42)
    base  = nn.Linear(64, 128, bias=False)
    layer = DoRALinear(base, r=4, alpha=8.0).to(torch.float32)
    with torch.no_grad():
        layer.lora_B.normal_(0, 0.01)
    W         = layer.base.weight
    W_adapted = W + (layer.lora_B @ layer.lora_A) * layer.scaling
    dir_norms = (W_adapted / W_adapted.norm(p=2, dim=1, keepdim=True)).norm(p=2, dim=1)
    ok        = torch.allclose(dir_norms, torch.ones_like(dir_norms), atol=1e-5)
    print(f"TEST 6 — Unit-direction per neuron    {PASS if ok else FAIL}")

test_unit_direction()


def test_llama_swap():
    try:
        from transformers import LlamaConfig, LlamaForCausalLM
        cfg = LlamaConfig(
            hidden_size=256,
            intermediate_size=512,
            num_hidden_layers=2,
            num_attention_heads=8,
            num_key_value_heads=2,
            max_position_embeddings=64,
            vocab_size=1000,
        )
        model  = LlamaForCausalLM(cfg)
        model  = apply_dora_to_llama(model, r=4, alpha=8.0)
        ok     = all(isinstance(layer.self_attn, DoRALlamaMHA) for layer in model.model.layers)
        counts = dora_param_count(model)
        print(f"TEST 7 — Llama attn layer swap        {PASS if ok else FAIL}")
        print(f"         trainable {counts['trainable']:,} / {counts['total']:,} params  ({counts['trainable_pct']}%)")
    except ImportError:
        print("TEST 7 — SKIPPED (transformers not installed)")

test_llama_swap()


def test_llama_forward():
    try:
        from transformers import LlamaConfig, LlamaForCausalLM
        cfg = LlamaConfig(
            hidden_size=256,
            intermediate_size=512,
            num_hidden_layers=2,
            num_attention_heads=8,
            num_key_value_heads=2,
            max_position_embeddings=64,
            vocab_size=1000,
        )
        model     = LlamaForCausalLM(cfg)
        model     = apply_dora_to_llama(model, r=4, alpha=8.0)
        model.eval()
        input_ids = torch.randint(0, 1000, (1, 16))
        with torch.no_grad():
            out = model(input_ids)
        ok = out.logits.shape == (1, 16, 1000)
        print(f"TEST 8 — Llama forward pass shape     {PASS if ok else FAIL}  {tuple(out.logits.shape)}")
    except ImportError:
        print("TEST 8 — SKIPPED (transformers not installed)")

test_llama_forward()

print("\nAll tests complete.")

Writing test_dora.py


In [43]:
if "dora" in sys.modules:
    del sys.modules["dora"]
%run /content/drive/MyDrive/DoRA/src/test_dora.py

Running on cpu (torch.float32)

TEST 1 — Trainable/frozen split       ✅ PASS
TEST 2 — Identity at init             ✅ PASS
TEST 3 — Magnitude init shape/value   ✅ PASS
TEST 4 — Output shape (2, 7, 128)          ✅ PASS
TEST 5 — Gradients (train/frozen)     ✅ PASS
TEST 6 — Unit-direction per neuron    ✅ PASS
TEST 7 — Llama attn layer swap        ✅ PASS
         trainable 1,314,304 / 1,641,984 params  (80.0437%)
TEST 8 — Llama forward pass shape     ✅ PASS  (1, 16, 1000)

All tests complete.


---
## End-to-End Training Pipeline

Everything below wires together `dora.py` (Person 1), `data.py` + `evaluate.py` (Person 2), and `inject.py` + `train.py` (Person 3) into a single training run on Colab.

All data, checkpoints, source files, model cache, and logs are stored on **Google Drive** so nothing is lost on runtime disconnect.

### Mount Google Drive & Create Directory Structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT   = "/content/drive/MyDrive/DoRA"
SRC_DIR      = f"{DRIVE_ROOT}/src"
DATA_DIR     = f"{DRIVE_ROOT}/data"
EVAL_DIR     = f"{DRIVE_ROOT}/eval_data"
CKPT_DIR     = f"{DRIVE_ROOT}/checkpoints"
MODEL_CACHE  = f"{DRIVE_ROOT}/model_cache"
LOG_DIR      = f"{DRIVE_ROOT}/logs"

for d in [SRC_DIR, DATA_DIR, EVAL_DIR, CKPT_DIR, MODEL_CACHE, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# Cache HF models on Drive so they persist across sessions
os.environ["HF_HOME"] = MODEL_CACHE
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE

print("Drive mounted. Directory structure:")
for d in [SRC_DIR, DATA_DIR, EVAL_DIR, CKPT_DIR, MODEL_CACHE, LOG_DIR]:
    print(f"  {d}")

### Copy `data.py` and `evaluate.py` to Drive (first time only)

Person 2's files aren't written by `%%writefile` cells, so upload them to `Drive/DoRA/src/` once via the sidebar, or run the cell below after uploading them to `/content/`.

In [ ]:
# First time: upload data.py and evaluate.py to /content/, then copy to Drive
import shutil
for f in ["data.py", "evaluate.py"]:
    src = f"/content/{f}"
    dst = f"{SRC_DIR}/{f}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"Copied {f} -> {dst}")
    elif os.path.exists(dst):
        print(f"{f} already on Drive")
    else:
        print(f"WARNING: {f} not found at {src} — upload it first")

### Write `inject.py` — Model Surgery

In [ ]:
%%writefile /content/drive/MyDrive/DoRA/src/inject.py
import re
from typing import List, Optional

import torch.nn as nn

from dora import DoRALinear, apply_dora_to_llama, dora_param_count


def _name_matches(name: str, target_names: List[str]) -> bool:
    """Return True if the module name ends with any of the target patterns."""
    for pattern in target_names:
        if name == pattern or name.endswith(f".{pattern}"):
            return True
    return False


def _set_submodule(model: nn.Module, key: str, new_module: nn.Module):
    tokens = key.split(".")
    parent = model
    for tok in tokens[:-1]:
        parent = getattr(parent, tok)
    setattr(parent, tokens[-1], new_module)


def inject_dora(
    model: nn.Module,
    target_names: List[str],
    r: int = 16,
    lora_alpha: float = 32.0,
    dropout: float = 0.0,
    use_llama_mha: bool = False,
) -> nn.Module:
    """
    Replace matching nn.Linear layers with DoRALinear adapters.

    Parameters
    ----------
    model : nn.Module
        The pretrained model to modify in-place.
    target_names : list[str]
        Suffixes to match against module names (e.g. ["q_proj", "v_proj"]).
    r : int
        LoRA rank.
    lora_alpha : float
        LoRA scaling factor.
    dropout : float
        Dropout rate applied after the adapted linear (unused by current
        DoRALinear — reserved for future extension).
    use_llama_mha : bool
        If True, use the full DoRALlamaMHA attention replacement instead
        of individual Linear swaps.  Recommended for Llama models.
    """
    if use_llama_mha:
        return apply_dora_to_llama(model, r=r, alpha=lora_alpha)

    replaced = []
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and _name_matches(name, target_names):
            dora_layer = DoRALinear(module, r=r, alpha=lora_alpha)
            _set_submodule(model, name, dora_layer)
            replaced.append(name)

    if not replaced:
        raise ValueError(
            f"No nn.Linear modules matched target_names={target_names}. "
            "Check the model architecture and target names."
        )

    print(f"[inject_dora] Replaced {len(replaced)} layers:")
    for name in replaced:
        print(f"  - {name}")

    return model


ADAPTER_KEYWORDS = {"lora_A", "lora_B", "magnitude"}


def get_trainable_params(model: nn.Module) -> list:
    """Return (name, param) pairs for DoRA adapter parameters only."""
    return [
        (n, p) for n, p in model.named_parameters()
        if p.requires_grad and any(kw in n for kw in ADAPTER_KEYWORDS)
    ]


def print_trainable_params(model: nn.Module) -> None:
    """Print count and percentage of trainable vs total parameters."""
    counts = dora_param_count(model)
    print(
        f"Trainable params: {counts['trainable']:,} / {counts['total']:,} "
        f"({counts['trainable_pct']}%) | Frozen: {counts['frozen']:,}"
    )

### Write `train.py` — Training Script

In [ ]:
%%writefile /content/drive/MyDrive/DoRA/src/train.py
"""
DoRA fine-tuning script for Llama 3.1 8B on commonsense reasoning tasks.

Usage
-----
    python train.py \
        --model_name meta-llama/Llama-3.1-8B \
        --data_path  ./data/train.json \
        --output_dir ./checkpoints \
        --r 16 --alpha 32 \
        --epochs 3 --lr 2e-4 --batch_size 4
"""

import argparse
import json
import os
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

from data import CommonsenseDataset, collate_fn, load_dataset_json
from inject import inject_dora, print_trainable_params, get_trainable_params
from evaluate import evaluate_task, load_dataset_json as load_eval_json


# ── CLI ──────────────────────────────────────────────────────────────────────

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="DoRA fine-tuning for Llama")

    # model / data
    p.add_argument("--model_name", type=str, default="meta-llama/Llama-3.1-8B")
    p.add_argument("--data_path", type=str, required=True,
                    help="Path to training JSON (list of instruction samples)")
    p.add_argument("--eval_path", type=str, default=None,
                    help="Path to eval JSON for end-of-epoch accuracy check")
    p.add_argument("--output_dir", type=str, default="./checkpoints")

    # DoRA hyper-params
    p.add_argument("--r", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32.0)
    p.add_argument("--target_names", nargs="+",
                    default=["q_proj", "k_proj", "v_proj", "o_proj"],
                    help="Linear layer name suffixes to replace with DoRA")

    # training hyper-params
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--batch_size", type=int, default=4)
    p.add_argument("--max_length", type=int, default=256)
    p.add_argument("--warmup_steps", type=int, default=100)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--grad_accum_steps", type=int, default=1)
    p.add_argument("--max_grad_norm", type=float, default=1.0)
    p.add_argument("--log_interval", type=int, default=10)

    # hardware
    p.add_argument("--dtype", type=str, default="bfloat16",
                    choices=["float32", "float16", "bfloat16"])
    p.add_argument("--device", type=str, default=None,
                    help="Force device (default: auto-detect)")

    return p.parse_args()


# ── helpers ──────────────────────────────────────────────────────────────────

DTYPE_MAP = {
    "float32": torch.float32,
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
}

ADAPTER_KEYWORDS = {"lora_A", "lora_B", "magnitude"}


def resolve_device(requested: str | None) -> torch.device:
    if requested:
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def save_adapter_checkpoint(model, path: Path, epoch: int, step: int):
    """Save only DoRA adapter weights (lora_A, lora_B, magnitude)."""
    adapter_state = {
        k: v.cpu()
        for k, v in model.state_dict().items()
        if any(kw in k for kw in ADAPTER_KEYWORDS)
    }
    ckpt = {
        "adapter_state_dict": adapter_state,
        "epoch": epoch,
        "step": step,
    }
    path.mkdir(parents=True, exist_ok=True)
    save_path = path / f"adapter_epoch{epoch}_step{step}.pt"
    torch.save(ckpt, save_path)
    print(f"[checkpoint] Saved {len(adapter_state)} tensors -> {save_path}")


def load_adapter_checkpoint(model, ckpt_path: str | Path):
    """Load DoRA adapter weights back into the model."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(
        ckpt["adapter_state_dict"], strict=False
    )
    print(f"[checkpoint] Loaded adapter from {ckpt_path} "
          f"(missing={len(missing)}, unexpected={len(unexpected)})")
    return ckpt.get("epoch", 0), ckpt.get("step", 0)


# ── main ─────────────────────────────────────────────────────────────────────

def main():
    args = parse_args()
    device = resolve_device(args.device)
    dtype = DTYPE_MAP[args.dtype]

    print(f"Device: {device}  |  dtype: {dtype}")
    print(f"Model : {args.model_name}")
    print(f"Rank  : {args.r}  |  Alpha: {args.alpha}")

    # ── tokenizer ────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # ── model ────────────────────────────────────────────────────────────
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=dtype,
        device_map=device if device.type == "cuda" else None,
    )

    # ── inject DoRA adapters (uses Llama-specific MHA swap) ──────────────
    model = inject_dora(
        model,
        target_names=args.target_names,
        r=args.r,
        lora_alpha=args.alpha,
        use_llama_mha=True,
    )
    print_trainable_params(model)

    if device.type != "cuda":
        model.to(device)

    # ── dataset & dataloader ─────────────────────────────────────────────
    raw_samples = load_dataset_json(args.data_path)
    dataset = CommonsenseDataset(raw_samples, tokenizer, max_length=args.max_length)
    dataloader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    # ── optimizer & scheduler ────────────────────────────────────────────
    trainable = [p for _, p in get_trainable_params(model)]
    optimizer = torch.optim.AdamW(
        trainable,
        lr=args.lr,
        weight_decay=args.weight_decay,
    )
    total_steps = (len(dataloader) // args.grad_accum_steps) * args.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=args.warmup_steps,
        num_training_steps=total_steps,
    )

    print(f"\nDataset  : {len(dataset):,} samples")
    print(f"Batches  : {len(dataloader):,} per epoch")
    print(f"Steps    : {total_steps:,} total ({args.epochs} epochs)")
    print()

    # ── training loop ────────────────────────────────────────────────────
    output_dir = Path(args.output_dir)
    global_step = 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for step, batch in enumerate(dataloader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss / args.grad_accum_steps
            loss.backward()

            if step % args.grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(trainable, args.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss += outputs.loss.item()

            if step % args.log_interval == 0:
                avg = epoch_loss / step
                lr_now = scheduler.get_last_lr()[0]
                elapsed = time.time() - t0
                print(
                    f"  epoch {epoch} | step {step}/{len(dataloader)} | "
                    f"loss {outputs.loss.item():.4f} | avg {avg:.4f} | "
                    f"lr {lr_now:.2e} | {elapsed:.0f}s"
                )

        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch} done — avg loss: {avg_loss:.4f}  "
              f"({time.time() - t0:.0f}s)")

        save_adapter_checkpoint(model, output_dir, epoch, global_step)

        # ── optional end-of-epoch eval ───────────────────────────────────
        if args.eval_path:
            eval_samples = load_eval_json(args.eval_path)
            acc = evaluate_task(model, tokenizer, eval_samples, batch_size=1)
            print(f"  eval accuracy: {acc * 100:.2f}%")

    print("\nTraining complete.")


if __name__ == "__main__":
    main()

### Prepare Data

Upload your commonsense training JSON to `Drive/DoRA/data/train.json` (via the Colab sidebar or any method below).
Each sample needs `instruction`, `output`, and optionally `input` keys.

In [ ]:
TRAIN_PATH = f"{DATA_DIR}/train.json"

# Option A: Upload via Colab sidebar into Drive > DoRA > data > train.json

# Option B: gdown (paste your file id)
# !gdown <FILE_ID> -O {TRAIN_PATH}

# Option C: HuggingFace Hub
# from huggingface_hub import hf_hub_download
# hf_hub_download(repo_id="...", filename="train.json", local_dir=DATA_DIR)

import os
assert os.path.exists(TRAIN_PATH), f"Upload train.json to {DATA_DIR} first!"
print(f"train.json found at {TRAIN_PATH}")

### GPU Check & Imports

In [ ]:
import sys, torch
sys.path.insert(0, SRC_DIR)

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

from dora import DoRALinear, apply_dora_to_llama, dora_param_count
from data import CommonsenseDataset, collate_fn, load_dataset_json
from inject import inject_dora, get_trainable_params, print_trainable_params
from evaluate import evaluate_task, load_dataset_json as load_eval_json
print("\nAll imports OK")

### Load Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
print("Model loaded")

### Inject DoRA Adapters

In [ ]:
RANK = 16
ALPHA = 32.0

model = inject_dora(
    model,
    target_names=["q_proj", "k_proj", "v_proj", "o_proj"],
    r=RANK,
    lora_alpha=ALPHA,
    use_llama_mha=True,
)
print_trainable_params(model)

### Build DataLoader

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4
MAX_LENGTH = 256

raw_samples = load_dataset_json(TRAIN_PATH)
print(f"Loaded {len(raw_samples)} training samples")

dataset = CommonsenseDataset(raw_samples, tokenizer, max_length=MAX_LENGTH)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)
print(f"{len(dataloader)} batches per epoch")

### Optimizer & Scheduler

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
LR = 2e-4
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 1

trainable = [p for _, p in get_trainable_params(model)]
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = (len(dataloader) // GRAD_ACCUM) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)
print(f"Total training steps: {total_steps}")

### Training Loop

In [ ]:
import time, json

LOG_EVERY = 10
MAX_GRAD_NORM = 1.0
ADAPTER_KEYWORDS = {"lora_A", "lora_B", "magnitude"}
device = torch.device("cuda")

log_path = f"{LOG_DIR}/train_log.jsonl"
log_file = open(log_path, "a")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(dataloader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += outputs.loss.item()

        if step % LOG_EVERY == 0:
            avg = epoch_loss / step
            lr_now = scheduler.get_last_lr()[0]
            print(
                f"  epoch {epoch} | step {step}/{len(dataloader)} | "
                f"loss {outputs.loss.item():.4f} | avg {avg:.4f} | "
                f"lr {lr_now:.2e} | {time.time()-t0:.0f}s"
            )
            log_file.write(json.dumps({
                "epoch": epoch, "step": step,
                "loss": outputs.loss.item(), "avg_loss": avg, "lr": lr_now,
            }) + "\n")
            log_file.flush()

    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch} done — avg loss: {avg_loss:.4f} ({time.time()-t0:.0f}s)")

    # save checkpoint to Drive (adapter weights only)
    adapter_state = {
        k: v.cpu() for k, v in model.state_dict().items()
        if any(kw in k for kw in ADAPTER_KEYWORDS)
    }
    ckpt_path = f"{CKPT_DIR}/adapter_epoch{epoch}.pt"
    torch.save({"adapter_state_dict": adapter_state, "epoch": epoch}, ckpt_path)
    print(f"  Saved {len(adapter_state)} adapter tensors -> {ckpt_path}")

log_file.close()
print(f"\nTraining complete. Logs at {log_path}")

### Evaluation

Evaluate on one or all 8 commonsense tasks. Update paths to point at your eval JSON files.

In [ ]:
from evaluate import run_all_tasks

# Single-task eval
# eval_samples = load_eval_json(f"{EVAL_DIR}/boolq/test.json")
# acc = evaluate_task(model, tokenizer, eval_samples, batch_size=4)
# print(f"BoolQ accuracy: {acc*100:.2f}%")

# All 8 tasks (needs EVAL_DIR/<task>/test.json for each task)
# results = run_all_tasks(model, tokenizer, EVAL_DIR, batch_size=4)

### Download Checkpoint

In [ ]:
# Checkpoints are already on Drive — no download needed!
# To download locally anyway:
from google.colab import files
files.download(f"{CKPT_DIR}/adapter_epoch{EPOCHS}.pt")

### Alternative: Run via `train.py` CLI

Instead of the cells above, you can run the full pipeline as a single command:

In [ ]:
# %cd /content/drive/MyDrive/DoRA/src
# !python train.py \
#     --model_name meta-llama/Llama-3.1-8B \
#     --data_path /content/drive/MyDrive/DoRA/data/train.json \
#     --output_dir /content/drive/MyDrive/DoRA/checkpoints \
#     --r 16 --alpha 32 \
#     --epochs 3 --lr 2e-4 --batch_size 4 \
#     --warmup_steps 100 --dtype bfloat16